# First steps with DigitalHub

This notebook supports the first steps with the platform:
* create a project
* create and run a function
* write artifacts

## Project initialization

Create a project: a dedicated space where we can manage functions, artifacts and executions.

Use the ``username`` as param to create a personal project in the shared space.

In [ ]:
import digitalhub as dh
import os

project = dh.get_or_create_project(f"{os.environ['USER']}-base")
project

## 1. Create a function and run it locally

A simple hello world script can be registered as a *function* and executed via sdk. We need to define:

* the source code
* the name of the function
* the `handler`: the function to be called




In [ ]:
# %%writefile "hellojob.py"

def hello():
    print("Hello Job!")

In [ ]:
hello()

So let's write a file and then define a function with the file as source

In [ ]:
%%writefile "hellojob.py"

def hello():
    print("Hello Job!")

In [ ]:
func = project.new_function(name="hello-job",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="hellojob.py",
                            handler="hello")

Now we should be able to run it locally (`local_execution=True`)

In [ ]:
run = func.run("job", wait=True, local_execution=True)

### 1.1 Remote execution

What about remote execution? Try with `local_execution=False` and check the console for progress.

In [ ]:
# remote execution: follow on the console the progress
run = func.run("job", wait=True, local_execution=False)



Did it work? Check results.

In [ ]:
run.results()

And the logs.

In [ ]:
run.logs()

Logs are base64 encoded, we need to decode to string.

In [ ]:
import base64

decoded_logs = base64.b64decode(run.logs()[1]["content"]).decode("utf-8")
print(decoded_logs)

### 1.2 Function Parameters
Functions should take parameters, let's add a parameter to hello world and then call with different inputs.

In [ ]:
%%writefile "hellojob.py"

def hello(name):
    print(f"Hello {name}!")

In [ ]:
func = project.new_function(name="hello-job",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="hellojob.py",
                            handler="hello")

In [ ]:
run = func.run("job", parameters={"name": "World"}, wait=True, local_execution=True)



### 1.3 Outputs
What about returning an output?



In [ ]:
%%writefile "hellojob.py"
from digitalhub_runtime_python import handler

# @handler(outputs=["greeting"])
def hello(name):
    text = f"Hello {name}!"
    print(text)
    return text

In [ ]:
func = project.new_function(name="hello-job",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="hellojob.py",
                            handler="hello")
run = func.run("job", parameters={"name": "World"}, wait=True, local_execution=True)


In [ ]:
run.results()

Good. This is the final version of the function. What about all the previous ones?
We can fetch them.

In [ ]:
versions = project.get_function_versions("hello-job")
print(versions)

we can run again a previous version, everything is still there. Take the first and run it.

In [ ]:
versions[-1].run("job", parameters={"name": "World"}, wait=True, local_execution=True)